In [0]:
ROOT = "/Workspace/Users/melaniepieterse1@gmail.com/Loan_Offer_Model"
CATALOG_PATH = f"{ROOT}/conf/base/catalog.yml"
import sys
sys.path.insert(0, ROOT)
from src.catalog_loader import Catalog
catalog = Catalog(spark, CATALOG_PATH)

customers_df = catalog.load("prep_customer")
prep_loans_df = catalog.load("loans")
prep_transactions_df = catalog.load("prep_transactions")

print(f"customers_df: {customers_df.count()} rows")
print(f"loans_df: {prep_loans_df.count()} rows")
print(f"transactions_df: {prep_transactions_df.count()} rows")

In [0]:
print(f"customers_df: {customers_df.count()} rows, {customers_df.select('ID').distinct().count()} unique IDs")
print(f"loans_df: {prep_loans_df.count()} rows, {prep_loans_df.select('ID').distinct().count()} unique IDs")
print(f"transactions_df: {prep_transactions_df.count()} rows, {prep_transactions_df.select('ID').distinct().count()} unique IDs")

In [0]:
from pyspark.sql.functions import when, col, count, datediff, lag, avg, mode, lit,  min, max, date_format, sum,weekofyear, max_by, std
from pyspark.sql.window import Window

In [0]:
# Determine the range of data  - data is availble over 4 months
date_range = prep_transactions_df.agg(min("DATE").alias("min_date"), max("DATE").alias("max_date"),)
display(date_range)

In [0]:
analysis_level = 'MONTH'  # 'WEEK' #'DAY

In [0]:
loans_df = prep_loans_df

w_outcome = Window.partitionBy("ID")
w_date = Window.partitionBy("ID").orderBy("DATE")

loans_df = loans_df.withColumn( "DECLINED_OUTCOME", count(when(col("OUTCOME") == "Declined", 1)).over(w_outcome)
).withColumn("SUCCESS_OUTCOME", count(when(col("OUTCOME") == "TakeUp", 1)).over(w_outcome)).withColumn("INTER_OFFER_TIME_DAYS", datediff(col("DATE"), lag("DATE").over(w_date)))

loans_df = loans_df.groupBy('ID').agg(count('*').alias('TOTAL_OFFERS'), avg('INTER_OFFER_TIME_DAYS').alias('AVG_INTER_OFFER_TIME_DAYS'), mode('DECLINED_OUTCOME').alias('DECLINED_OUTCOMES'), mode('SUCCESS_OUTCOME').alias('SUCCESS_OUTCOME'),avg('AMOUNT').alias('AVG_AMOUNT_OFFERED'), max('DATE').alias('LAST_OFFER_DATE'))

loans_df = loans_df.withColumn('DAYS_SINCE_LAST_OFFER', datediff(lit('2022-08-30'), col('LAST_OFFER_DATE'))).drop('LAST_OFFER_DATE')

loans_df = loans_df.withColumn('LABEL', when(col('SUCCESS_OUTCOME') > 0, 1).otherwise(0))

display(loans_df)


In [0]:
transactions_df = prep_transactions_df.join(customers_df, on='ID', how ='full')

transactions_df = transactions_df.withColumn('MONTH', date_format("DATE", "MM")).withColumn('WEEK', weekofyear("DATE")).withColumn('DAY', date_format("DATE", "dd"))


# Count the number of postive and negative analysis_level end balances. -all balances result in positive.

analysis_level_transactions = transactions_df.groupBy('ID', analysis_level, 'GENDER', 'AGE', 'INCOME').agg(max_by('BALANCE', '__index_level_0__').alias('END_BALANCE'),
    sum(when(col('AMOUNT') < 0, 1).otherwise(0)).alias('TRANS_COUNT'),
    sum(when(col('AMOUNT') > 0, 1).otherwise(0)).alias('INCOME_COUNT'),
    avg(when(col('AMOUNT') < 0, col('AMOUNT')).otherwise(lit(None))).alias('AVG_TRANS_AMOUNT'),
    sum(when(col('AMOUNT') > 0, col('AMOUNT')).otherwise(lit(None))).alias('TOTAL_INCOME'),
    sum(when(col('AMOUNT') < 0, col('AMOUNT')).otherwise(lit(None))).alias('TOTAL_EXPENSES'),
    avg(when(col('AMOUNT') == col('INCOME'), col('DAY').cast('int')).otherwise(lit(None))).alias('INCOME_DAY'),
    sum(when((col('AMOUNT') < 0) & (col('BALANCE') < 0), 1).otherwise(0)).alias('OVERDRAFT_SPENT'),
    sum(when((col('AMOUNT') < 0) & (col('BALANCE') < 0), col('AMOUNT')).otherwise(lit(None))).alias('OVERDRAFT_AMOUNT'))

analysis_level_transactions = analysis_level_transactions.withColumn('ADD_INCOME', when(col('TOTAL_INCOME') == 0, lit(None)).otherwise(when(col('TOTAL_INCOME') > col('INCOME'), col('TOTAL_INCOME') - col('INCOME')).otherwise(0))).withColumn('LIVING_MEANS', when(col('TOTAL_INCOME') == 0, lit(None)).otherwise(col('TOTAL_INCOME') - col('TOTAL_EXPENSES'))).withColumn('INCOME_SPENT_%', when(col('TOTAL_INCOME') == 0, lit(None)).otherwise((-col('TOTAL_EXPENSES') / col('TOTAL_INCOME')) * 100))

display(analysis_level_transactions)


transaction_summary = analysis_level_transactions.groupBy('ID','GENDER','AGE','INCOME').agg(sum(when(col('END_BALANCE') > 0, 1).otherwise(0)).alias('POSITIVE_END_BALANCE'),sum(when(col('END_BALANCE') < 0, 1).otherwise(0)).alias('NEGATIVE_END_BALANCE'), avg('END_BALANCE').alias('AVG_MONTHLY_END_BALANCE'), std('END_BALANCE').alias('STD_MONTHLY_END_BALANCE'),avg('TRANS_COUNT').alias('AVG_TRANS_COUNT'),std('TRANS_COUNT').alias('STD_TRANS_COUNT'),avg('INCOME_COUNT').alias('AVG_INCOME_COUNT'),std('INCOME_COUNT').alias('STD_INCOME_COUNT'),avg('AVG_TRANS_AMOUNT').alias('AVG_TRANS_AMOUNT'),std('AVG_TRANS_AMOUNT').alias('STD_TRANS_AMOUNT'),avg('TOTAL_INCOME').alias('AVG_TOTAL_INCOME'),std('TOTAL_INCOME').alias('STD_TOTAL_INCOME'),avg('TOTAL_EXPENSES').alias('AVG_TOTAL_EXPENSES'),std('TOTAL_EXPENSES').alias('STD_TOTAL_EXPENSES'),mode('INCOME_DAY').alias('INCOME_DAY'),avg('OVERDRAFT_AMOUNT').alias('AVG_OVERDRAFT_SPENT'),sum('OVERDRAFT_SPENT').alias('OVERDRAFT_SPENT'),avg('ADD_INCOME').alias('AVG_ADD_INCOME'),sum('ADD_INCOME').alias('ADD_INCOME'), avg('LIVING_MEANS').alias('LIVING_MEANS'),avg('INCOME_SPENT_%').alias('INCOME_SPENT_%')).fillna(0)

display(transaction_summary)

In [0]:
final_df = transaction_summary.join(loans_df, on ="ID", how = 'outer').fillna(0)
display(final_df)


Databricks visualization. Run in Databricks to view.